# Cottonwoods snowfall: exploration
Run top to bottom. Sections 1-3 need no internet. Sections 4-5 need network access and the setup in the README
(SNOTEL triplets filled in `config/stations.yaml`, Herbie/cfgrib/eccodes installed).

In [ ]:
import sys
from pathlib import Path

# make `import cottonwoods` work even if the package isn't pip-installed (run from notebooks/ or the repo root)
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cottonwoods import slr, regimes, journal
from cottonwoods.config import load_config, PROCESSED_DIR, RAW_DIR, ensure_dirs

pd.set_option("display.width", 160)
cfg = load_config()
ensure_dirs()

## 1. Journal vs actuals (offline)

In [ ]:
fc = journal.score_journal()
have = fc.dropna(subset=["actual_in"])
have[["storm_id", "model", "low_in", "high_in", "actual_in", "error_in", "hit", "notes"]]

In [ ]:
clean = have[~have.open_ended]
by_model = clean.groupby("model")["error_in"].agg(["count", "mean"]).round(1).sort_values("mean")
display(by_model)

ax = by_model["mean"].plot.barh(figsize=(6, 4))
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("mean error, inches (midpoint minus actual)")
ax.set_title("Journal forecast bias by model (few storms: a hint, not a result)")
plt.tight_layout()

In [ ]:
# Error as a fraction of the actual helps compare small and large storms
ratio = clean.pivot_table(index="storm_id", columns="model", values="ratio").round(2)
ratio

## 2. Kuchera vs a fixed 10:1 (offline)
The point from your journal: fixed 10:1 overestimates warm storms and underestimates cold ones.

In [ ]:
tmax_c = np.arange(-20, 6, 1.0)
tmax_k = tmax_c + 273.15
plt.figure(figsize=(6, 4))
plt.plot(tmax_c, slr.kuchera_slr(tmax_k), label="Kuchera")
plt.axhline(10, color="gray", ls="--", label="fixed 10:1")
plt.xlabel("Tmax in 850-500 mb layer (C)")
plt.ylabel("SLR")
plt.legend(); plt.grid(alpha=0.3)
plt.title("Kuchera SLR vs layer-max temperature")
plt.tight_layout()

In [ ]:
# Why per-step matters: same total QPF, different timing of warm vs cold air
qpf = pd.Series([0.20, 0.20, 0.10, 0.10])                  # inches liquid per step
tmax_k = pd.Series([272.5, 270.0, 266.0, 264.0])           # warm start, cold finish
per_step = slr.kuchera_slr(tmax_k.values)
per_step_total = (qpf * per_step).sum()
lumped_total = qpf.sum() * slr.kuchera_slr(tmax_k.mean())  # the common mistake
print(f"per-step: {per_step_total:.1f} in   lumped: {lumped_total:.1f} in")

## 3. Regime labels (offline demo)
Replace the example values with ERA5 or model-analysis event means near the Wasatch once you have them.

In [ ]:
examples = {
    "warm SW storm (like 2/16a)": dict(u=12.0, v=10.0, t700=-6.0, lake=8.0),
    "cold NW storm (like 2/16b)": dict(u=14.0, v=-8.0, t700=-14.0, lake=18.0),
}
for name, e in examples.items():
    d = float(regimes.wind_dir_deg(e["u"], e["v"]))
    print(name, "->", round(d), "deg", regimes.label_storm(d, e["t700"], e["lake"], cfg["regimes"]))

## 4. Observed SNOTEL (needs network + triplets)
First fill in triplets: run the cell below, then edit `config/stations.yaml`.

In [ ]:
from cottonwoods.obs import snotel
for q in ["Alta", "Snowbird", "Brighton"]:
    print(f"== {q}")
    print(snotel.find_stations(q).to_string(index=False))

In [ ]:
sites = {k: v["triplet"] for k, v in cfg["snotel"].items() if v["triplet"]}
assert sites, "Set SNOTEL triplets in config/stations.yaml first"

storms = journal.load_storms()
s = storms[storms.storm_id == "2024-12-08"].iloc[0]
begin = (s.start_date - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
end = (s.end_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
obs = snotel.fetch(list(sites.values()), begin, end)
obs.head()

In [ ]:
swe = obs[obs.element == "WTEQ"].pivot(index="time", columns="station", values="value")
swe.plot(figsize=(8, 4), title=f"SNOTEL SWE, storm {s.storm_id} (journal actual: {s.actual_in} in)")
plt.ylabel("SWE (in)"); plt.tight_layout()

## 5. HRRR + Kuchera for one journal forecast (needs network, Herbie, cfgrib, eccodes)
Untested against live data: expect to adjust variable names on first run.

In [ ]:
from cottonwoods.models.hrrr import pull_run

fcs = journal.load_forecasts()
row = fcs[(fcs.storm_id == "2024-12-08") & (fcs.model == "HRRR")].iloc[0]
init = row.init_utc.tz_localize(None).to_pydatetime()
df = pull_run(init, range(1, 49), cfg["points"], save_dir=str(RAW_DIR / "herbie"))
df.head()

In [ ]:
tcols = {int(c.split("_")[1]): c for c in df.columns if c.startswith("t_")}
temps = df[[tcols[l] for l in sorted(tcols)]].copy()
temps.columns = sorted(tcols)
k = cfg["kuchera"]
df["tmax_k"] = slr.layer_tmax(temps, k["bottom_level_hpa"], k["top_level_hpa"])
df["slr_kuchera"] = slr.kuchera_slr(df["tmax_k"].values, k["slr_floor"], k["slr_cap"])
df["snow_in"] = slr.qpf_mm_to_in(df["qpf_mm"]) * df["slr_kuchera"]

alta = df[df.point == "alta"].set_index("valid_end")
alta["snow_in"].cumsum().plot(figsize=(8, 4), title=f"HRRR {init:%Y-%m-%d %H}z, Alta point, cumulative Kuchera snowfall")
plt.ylabel("inches"); plt.tight_layout()
print("model total (in):", round(alta["snow_in"].sum(), 1), "| journal actual:", s.actual_in)